# Multi-tool Agent: External APIs + Tools

This demo integrates three kinds of capability into one Microsoft Agent Framework agent:

- **Custom function tools** that call public REST APIs with `httpx`
- A **hosted MCP tool** backed by Microsoft Learn
- Model-driven routing, so the agent chooses the right tool for each request

The public APIs used here do not require API keys: Open-Meteo for geocoding/weather and REST Countries for country facts.

## 1. Install dependencies

In [ ]:
%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2 httpx

## 2. Load Foundry configuration

Authenticate first with `az login`. Keep deployment-specific values in `.env`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

if not project_endpoint or not model:
    raise ValueError("Set AI_FOUNDRY_PROJECT_ENDPOINT and AI_FOUNDRY_DEPLOYMENT_NAME in .env")

print("Project endpoint:", project_endpoint)
print("Model deployment:", model)

## 3. Build external API function tools

These functions own HTTP concerns such as URLs, parameters, timeouts, status checks, and response shaping. The agent only sees clean typed tool schemas and compact JSON results.

In [ ]:
import httpx

HTTP_TIMEOUT = 15.0

async def get_current_weather(city: str) -> dict:
    """Get the current weather for a city, including temperature, wind speed, and weather code."""
    print(f"TOOL CALL -> get_current_weather(city={city!r})")
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT) as client:
        geo_response = await client.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en", "format": "json"},
        )
        geo_response.raise_for_status()
        locations = geo_response.json().get("results", [])
        if not locations:
            return {"error": f"No location found for {city}"}

        location = locations[0]
        weather_response = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "current": "temperature_2m,apparent_temperature,weather_code,wind_speed_10m",
                "timezone": "auto",
            },
        )
        weather_response.raise_for_status()
        payload = weather_response.json()
        return {
            "location": f"{location['name']}, {location.get('country', '')}",
            "timezone": payload.get("timezone"),
            "current": payload.get("current", {}),
            "units": payload.get("current_units", {}),
        }

async def get_country_facts(country_name: str) -> dict:
    """Get official country facts such as capital, region, population, currencies, and languages."""
    print(f"TOOL CALL -> get_country_facts(country_name={country_name!r})")
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT) as client:
        response = await client.get(
            f"https://restcountries.com/v3.1/name/{country_name}",
            params={"fullText": "true"},
        )
        if response.status_code == 404:
            return {"error": f"No country found for {country_name}"}
        response.raise_for_status()
        country = response.json()[0]
        return {
            "official_name": country.get("name", {}).get("official"),
            "capital": country.get("capital", []),
            "region": country.get("region"),
            "population": country.get("population"),
            "currencies": country.get("currencies", {}),
            "languages": country.get("languages", {}),
        }

## 4. Test API tools directly

Testing tools independently separates API/integration problems from agent reasoning problems.

In [ ]:
weather_test = await get_current_weather("Bengaluru")
country_test = await get_country_facts("India")

print("Weather API test:", weather_test)
print("Country API test:", country_test)

## 5. Register a hosted MCP tool

The Microsoft Learn MCP server exposes documentation tools remotely. `approval_mode="never_require"` keeps this classroom demo flowing; use an approval policy appropriate to your production risk.

In [ ]:
from agent_framework import HostedMCPTool

microsoft_learn = HostedMCPTool(
    name="Microsoft Learn",
    description="Search official Microsoft Learn documentation.",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)

## 6. Create the multi-tool agent

The agent receives all tools at creation time. The model reads their schemas and descriptions, then decides whether to call one tool, several tools, or none.

In [ ]:
from agent_framework.azure import AzureAIClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
conversation = await openai_client.conversations.create()

agent_client = AzureAIClient(
    project_client=project_client,
    conversation_id=conversation.id,
    model_deployment_name=model,
)

agent = agent_client.create_agent(
    name="External-API-and-Tools-Agent",
    description="Routes requests to weather, country facts, and Microsoft Learn tools.",
    instructions=(
        "Use tools whenever the user asks for current weather, country facts, or Microsoft product documentation. "
        "For mixed requests, call every relevant tool and synthesize the results. State which sources/tools were used. "
        "Do not invent API results."
    ),
    tools=[get_current_weather, get_country_facts, microsoft_learn],
)

print("Agent created with conversation:", conversation.id)

## 7. Run single-tool and multi-tool requests

In [ ]:
response = await agent.run("What is the current weather in Seattle?")
print(response.text)

In [ ]:
response = await agent.run(
    "I am visiting Japan. Give me the current weather in Tokyo plus Japan's capital, currency, population, and languages."
)
print(response.text)

In [ ]:
response = await agent.run(
    "Using official Microsoft Learn documentation, explain how Azure CLI authentication works for Python apps."
)
print(response.text)

## 8. Stream a response

Streaming improves the user experience for slower external calls and longer synthesized answers.

In [ ]:
async for update in agent.run_stream(
    "Compare the current weather in London with key facts about the United Kingdom."
):
    if update.text:
        print(update.text, end="", flush=True)

## 9. Production hardening checklist

- Store API secrets in Key Vault or environment variables, never in prompts.
- Validate tool arguments and restrict tools to the minimum required capabilities.
- Add retries, rate-limit handling, caching, telemetry, and circuit breakers around APIs.
- Require human approval for tools that write data, spend money, or affect infrastructure.
- Return compact structured results so the model receives only the data it needs.

## 10. Cleanup

In [ ]:
await agent_client.close()
await project_client.close()
await credential.close()
print("Closed Azure clients.")